# Somo la 12 - Kupunguza Historia ya Mazungumzo kwa kutumia Agent Scratchpad  

Daftari hili linaonesha jinsi ya kusimamia muktadha katika mazungumzo marefu kwa kutumia Microsoft Agent Framework. Kadri mazungumzo yanavyoongezeka, idadi ya tokeni huongezeka — hatimaye kuizidi dirisha la muktadha la modeli. Tunashughulikia hili kwa **kiolezo cha muhtasari wa muktadha** na **agent scratchpad** kwa kumbukumbu endelevu.  

## Utajifunza Nini:  
1. **Kwa Nini Usimamizi wa Muktadha ni Muhimu**: Kuelewa mipaka ya tokeni na dirisha za muktadha  
2. **Wakala Wanaojua Muktadha**: Kujenga mawakala wanaosimamia muktadha wa mazungumzo yao wenyewe  
3. **Kiolezo cha Muhtasari wa Muktadha**: Kutumia zana za kufupisha historia ya mazungumzo  
4. **Agent Scratchpad**: Kumbukumbu endelevu inayodumu baada ya kupunguzwa kwa muktadha  

## Masharti ya Awali:  
- Utekelezaji wa Azure OpenAI na mipangilio ya mazingira imewekwa  
- Uelewa wa dhana za msingi za wakala kutoka masomo yaliyopita  


## Usanidi


In [ ]:
%pip install agent-framework azure-ai-projects azure-identity python-dotenv --quiet

In [ ]:
import os
import asyncio
import dotenv
from datetime import datetime
from pathlib import Path

from agent_framework import tool
from agent_framework.foundry import FoundryChatClient
from azure.identity import DefaultAzureCredential

In [ ]:
dotenv.load_dotenv()

endpoint = os.getenv("AZURE_AI_PROJECT_ENDPOINT")
deployment_name = os.getenv("AZURE_AI_MODEL_DEPLOYMENT_NAME")

missing = [k for k, v in {
    "AZURE_AI_PROJECT_ENDPOINT": endpoint,
    "AZURE_AI_MODEL_DEPLOYMENT_NAME": deployment_name
}.items() if not v]

if missing:
    raise ValueError(
        f"Missing required environment variables: {', '.join(missing)}. "
        "Please set them as environment variables (e.g., in your .env file or shell environment)."
    )

# Create the Microsoft Foundry client
client = FoundryChatClient(
    project_endpoint=endpoint,
    model=deployment_name,
    credential=DefaultAzureCredential()
)

print("Microsoft Foundry client configured")

## Kwa Nini Usimamizi wa Muktadha ni Muhimu

Kila LLM ina **dirisha la muktadha** linaloweza kushughulikia idadi ya tokeni kwa ombi moja. Wakati mazungumzo ya mizunguko mingi yanaendelea:

- **Idadi ya tokeni huongezeka kwa msimamo wa mstari moja** na kila ujumbe wa mtumiaji na jibu la msaidizi.
- **Tokeni za ombi ndio zinazochangia gharama kubwa** kwa sababu historia yote hupelekwa tena kila mzunguko.
- Hatimaye mazungumzo **huvuka dirisha la muktadha** na mfano hutakataka au kupata makosa.

### Mikakati ya Kusimamia Muktadha

| Mkakati | Jinsi Inavyofanya Kazi | Mbadala |
|---|---|---|
| **Kukata** | Acha ujumbe wa zamani zaidi | Hupoteza muktadha wa awali |
| **Muhtasari** | Fupisha ujumbe wa zamani kuwa muhtasari | Maelezo mengine hupotea, lakini pointi muhimu zinabaki |
| **Kiandikio / Kumbukumbu ya Nje** | Hifadhi mambo muhimu nje ya mazungumzo | Inahitaji kuitwa zana, lakini huishi baada ya kupunguzwa |

Katika daftari hili tunachanganya **muhtasari** pamoja na **zana ya kiandikio** ili wakiweze kudumisha mfuatano hata mazungumzo yakiwa mafupi.


## Kuunda Wakala Anaye Fahamu Muktadha


In [ ]:
agent = client.as_agent(
    name="ContextAwareAgent",
    instructions="""You are a helpful travel planning assistant with excellent memory management.
When conversations get long:
1. Summarize previous context into key points
2. Track user preferences mentioned earlier
3. Reference previous decisions without repeating full details
Always maintain continuity while being concise.""",
)

print("Context-aware travel planning agent created")

## Kuiga Mazungumzo Marefu

Hebu tupitie mazungumzo ya mizunguko mingi kuona jinsi muktadha unavyokusanyika. Wakala anapaswa kuhifadhi maelezo muhimu (mapendeleo, bajeti, tarehe za kusafiri) katika mizunguko yote na kuonyesha mfuatano.


In [ ]:
session = agent.create_session()

# Turn 1 - Initial preferences
response = await agent.run("I'm planning a trip to Japan. I love sushi, temples, and photography.", session=session)
print(f"Turn 1: {response}\n")

# Turn 2 - More details
response = await agent.run("My budget is $3000 and I'll be traveling solo for 10 days in April.", session=session)
print(f"Turn 2: {response}\n")

# Turn 3 - Test context retention
response = await agent.run("Based on everything I've told you so far, what's the one thing you'd recommend I not miss?", session=session)
print(f"Turn 3: {response}\n")

Angalia jinsi wakala anavyoendelea kuweka muktadha kutoka kwa mizunguko ya awali — anajua kuhusu Japan, sushi, mahekalu, upigaji picha, bajeti ya $3000, safari binafsi, na kipindi cha Aprili. Katika mazungumzo mafupi hii hufanya kazi vizuri, lakini kadri mazungumzo yanavyozidi muktadha mzima unakuwa gharama kubwa kutumwa tena.

Tuendelee na mazungumzo kwa mizunguko zaidi kuona jinsi muktadha unavyokusanywa:


In [ ]:
# Turn 4 - Expand the conversation
response = await agent.run("What about accommodation? I prefer traditional Japanese inns.", session=session)
print(f"Turn 4: {response}\n")

# Turn 5 - Change of plans
response = await agent.run("Actually, I've changed my mind about the dates. I'll go in October instead for the autumn colors.", session=session)
print(f"Turn 5: {response}\n")

# Turn 6 - Test retention after change
response = await agent.run("Summarize my complete travel plan so far — destination, budget, duration, interests, accommodation, and timing.", session=session)
print(f"Turn 6: {response}\n")

## Mfano wa Muhtasari wa Muktadha

Kadri mazungumzo yanavyoongezeka, tunaweza kutumia **zana ya muhtasari** kupunguza muktadha uliokusanywa katika muundo wa kifupi. Wakala huita zana hii kurekodi mapendeleo muhimu ili hata kama ujumbe wa zamani utafutwa, taarifa muhimu zihakikishwe.

Mfano huu ni jiwe la msingi kwa upunguzaji wa historia wenye ustadi zaidi:
1. Wakala hutambua ukweli muhimu kutoka kwenye mazungumzo
2. Hudai zana ya muhtasari kuziweka uhifadhi
3. Ujumbe wa zamani unaweza kufutwa kwa usalama kwa sababu muhtasari unakamata yale yanayohitajika

Hapo chini tunaelezea zana `summarize_preferences` ambayo wakala anaweza kuitumia kurekodi muhtasari mfupi wa kile alichojifunza.


In [ ]:
@tool(approval_mode="never_require")
def summarize_preferences(conversation_notes: str) -> str:
    """Summarize accumulated user preferences into a compact format."""
    return f"[SUMMARY] User preferences recorded: {conversation_notes}"


# Create an enhanced agent with the summarization tool
summarizing_agent = client.as_agent(
    name="SummarizingTravelAgent",
    instructions="""You are a helpful travel planning assistant that actively manages conversation context.

CONTEXT MANAGEMENT RULES:
1. After gathering several user preferences, call summarize_preferences() to record a compact summary
2. When the user asks you to recall details, reference your recorded summaries
3. Keep responses concise — avoid restating the entire history

PLANNING PROCESS:
1. Gather user preferences (destination, budget, dates, interests)
2. Summarize preferences using the tool
3. Create recommendations based on the summary
4. Update the summary when preferences change""",
    tools=[summarize_preferences],
)

print("Summarizing travel agent created with context tools")

In [ ]:
# Demonstrate the summarization pattern
summary_session = summarizing_agent.create_session()

# Provide a batch of preferences
response = await summarizing_agent.run(
    "I want to visit Greece. I love seafood, history, and island hopping. "
    "Budget is $4000 for two weeks. Traveling with my partner in June. "
    "Please record these preferences using your summarization tool.",
    session=summary_session,
)
print(f"Agent: {response}\n")

# Ask the agent to use the recorded context
response = await summarizing_agent.run(
    "Now, based on what you've recorded, suggest the top 3 islands we should visit.",
    session=summary_session,
)
print(f"Agent: {response}\n")

## Muhtasari

Katika somo hili ulijifunza jinsi ya kusimamia muktadha katika mazungumzo ya mawakala yanayoendelea kwa muda mrefu kwa kutumia Microsoft Agent Framework:

### Dhana Muhimu
- **Dirisha za muktadha ni finitu** — kila tokeni katika historia ya mazungumzo inagharimu pesa na kuhesabiwa kuelekea kikomo.
- **Zana za muhtasari** zinamruhusu wakala kufupisha muktadha uliokusanywa kuwa muhtasari mfupi, kupunguza matumizi ya tokeni huku zikiweka taarifa muhimu.
- **Karatasi za mawakala** hutoa kumbukumbu ya nje inayodumu ambayo haipotei hata baada ya kupunguzwa kwa mazungumzo.

### Uliyotengeneza
- **Mwakala anayejua muktadha** anayehifadhi mfululizo wa mazungumzo yenye mizunguko mingi
- **Zana ya muhtasari** (`summarize_preferences`) inayorekodi maelezo muhimu ya mtumiaji kwa muundo mfupi
- **Mazungumzo yenye mizunguko mingi** yanayoonyesha uhifadhi wa muktadha na usimamizi wa mabadiliko

### Matumizi Halisi
- **Roboti wa Huduma kwa Wateja**: Kumbuka mapendeleo katika vikao virefu vya msaada
- **Msaidizi Binafsi**: Fuata miradi inayoendelea bila kufafanua tena muktadha
- **Walimu wa Elimu**: Hifadhi maendeleo ya mwanafunzi katika mwingiliano mingi

### Hatua Zifuatazo
- Tekeleza zana kamili ya karatasi za mawakala yenye uhifadhi wa faili
- Ongeza ukataji wa historia kiotomatiki baada ya muhtasari
- Changanya na hifadhidata za vector kwa ajili ya utafutaji wa kumbukumbu ya maana
- Tengeneza mawakala wanaoweza kuendelea na mazungumzo siku kadhaa baadaye na muktadha kamili


---

<!-- CO-OP TRANSLATOR DISCLAIMER START -->
**Kionyozo**:
Hati hii imetafsiriwa kwa kutumia huduma ya tafsiri ya AI [Co-op Translator](https://github.com/Azure/co-op-translator). Ingawa tunajitahidi kupata usahihi, tafadhali fahamu kwamba tafsiri za kiotomatiki zinaweza kuwa na makosa au upungufu wa usahihi. Hati ya asili katika lugha yake halisi inapaswa kuchukuliwa kama chanzo cha mamlaka. Kwa taarifa muhimu, tafsiri ya kitaalamu inayofanywa na binadamu inapendekezwa. Hatutojibu kwa kuelewa vibaya au tafsiri potofu zinazotokea kutokana na matumizi ya tafsiri hii.
<!-- CO-OP TRANSLATOR DISCLAIMER END -->
